# Carbon-Aware FL: FedAvg baseline on Colab

Simulated clients, MNIST or CIFAR-10, FedAvg, IID or Dirichlet non-IID splits, with round-level and client-level CSV logs.
No scheduler or RL yet; this is the reference run everything later is compared against.

**Runtime:** works on CPU; for speed choose *Runtime → Change runtime type → T4 GPU*.
**Energy and carbon:** every run also tracks the energy used by this machine (GPU via NVML, CPU via RAPL if exposed,
otherwise a TDP-based estimate) and converts it to gCO2e using a grid carbon intensity. The default of 475 gCO2e/kWh is an
assumed global average, so set `--carbon-intensity` to your region's value for a better number.


In [ ]:
# Clone (or update) the repository
REPO_URL = "https://github.com/Harshil089/Carbon-Aware-Federated-Learning.git"
BRANCH = "fl-baseline"  # switch to "main" once fl-baseline is merged

import os
if not os.path.exists("/content/Carbon-Aware-Federated-Learning"):
    !git clone -b {BRANCH} {REPO_URL} /content/Carbon-Aware-Federated-Learning
else:
    !git -C /content/Carbon-Aware-Federated-Learning pull
%cd /content/Carbon-Aware-Federated-Learning

In [ ]:
!pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# Unit tests for FedAvg and the partitioners (a few seconds)
!python -m pytest -q tests

## Run the baseline

Defaults live in `configs/experiment.yaml` (MNIST, small CNN, 5 clients, all participate, 20 rounds).
Flags override them: `--partition iid|dirichlet`, `--alpha`, `--rounds`, `--clients-per-round`, `--dataset fashion_mnist`, `--model mlp`, `--seed`.

In [ ]:
# IID
!python experiments/baseline/run_baseline.py --partition iid --rounds 20

In [ ]:
# Non-IID (Dirichlet). Smaller alpha = more skewed clients.
!python experiments/baseline/run_baseline.py --partition dirichlet --alpha 0.5 --rounds 20

## CIFAR-10: 50 clients, 10 per round

The harder benchmark. `configs/experiment_cifar10.yaml` samples 50 clients from low-end / mid-range / high-end tiers
(`configs/clients_pool50.yaml`), picks 10 at random each round, and trains a ~1.1M-parameter CNN with random flips.
On a T4 expect roughly 10-15 minutes per 150-round run (an estimate; check the printed timings).
Try `--alpha 0.1` (strongly non-IID), `0.5` and `10` (near-IID).


In [ ]:
# CIFAR-10, strongly non-IID (downloads ~170 MB the first time)
!python experiments/baseline/run_baseline.py --config configs/experiment_cifar10.yaml --alpha 0.1


## Inspect results

In [ ]:
import glob, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

runs = sorted(glob.glob("results/*/summary.json"))
summaries = pd.DataFrame([json.load(open(p)) for p in runs])
display(summaries[["experiment_id", "dataset", "partition", "alpha", "rounds", "final_accuracy", "best_accuracy",
                   "sim_total_latency_s", "sim_total_carbon_gco2e", "total_wall_time_s", "device"]])

fig, ax = plt.subplots(figsize=(7, 4))
for p in runs:
    run_dir = os.path.dirname(p)
    df = pd.read_csv(f"{run_dir}/rounds.csv")
    ax.plot(df["round"], df["global_accuracy"], marker="o", label=os.path.basename(run_dir))
ax.set(xlabel="Round", ylabel="Test accuracy", title="FedAvg global accuracy")
ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.show()

latest = os.path.dirname(max(runs, key=os.path.getmtime))  # most recent run
display(Image(f"{latest}/class_distribution.png"))
pd.read_csv(f"{latest}/rounds.csv").head()

## Energy and carbon
Two different things are logged:

- **Simulated federation** (`sim_*` columns in `rounds.csv` and `clients.csv`): each selected client is a separate device with its own
  speed, power, bandwidth and grid carbon intensity from its profile. Round latency is the slowest client (download + train + upload)
  plus aggregation; carbon uses each client's own grid. These are the objectives the scheduler will optimise. All inputs are placeholders.
- **This Colab VM** (`energy.json`, `round_energy_kwh` etc.): measured/estimated energy of the machine running the simulation, at one
  assumed grid intensity. Scope is CPU + GPU of this VM only (no DRAM, storage, network, cooling or embodied carbon).


In [ ]:
rows = []
for p in runs:
    run_dir = os.path.dirname(p)
    e = json.load(open(f"{run_dir}/energy.json"))
    rows.append({
        "run": os.path.basename(run_dir)[:48],
        "sim_latency_min": round(json.load(open(p)).get("sim_total_latency_s", float("nan")) / 60, 1),
        "sim_Wh": round(json.load(open(p)).get("sim_total_energy_kwh", float("nan")) * 1000, 2),
        "sim_gCO2e": round(json.load(open(p)).get("sim_total_carbon_gco2e", float("nan")), 2),
        "energy_Wh": round(e["energy_kwh"] * 1000, 3),
        "cpu_Wh": round(e["cpu_energy_kwh"] * 1000, 3),
        "gpu_Wh": round(e["gpu_energy_kwh"] * 1000, 3),
        "avg_power_W": round(e["avg_power_w"], 1),
        "gCO2e": round(e["carbon_gco2e"], 3),
        "g/kWh": e["carbon_intensity_gco2e_kwh"],
        "cpu_source": e["cpu_source"],
        "gpu_source": e["gpu_source"],
    })
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(7, 4))
for p in runs:
    run_dir = os.path.dirname(p)
    df = pd.read_csv(f"{run_dir}/rounds.csv")
    ax.plot(df["sim_cumulative_carbon_gco2e"], df["global_accuracy"], marker="o", label=os.path.basename(run_dir)[:40])
ax.set(xlabel="Simulated cumulative emissions (gCO2e)", ylabel="Test accuracy", title="Accuracy vs simulated carbon")
ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.show()

## Save results
Colab storage is wiped when the runtime ends. Download a zip, or mount Drive and copy `results/`.

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")

# Or keep them in Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r results /content/drive/MyDrive/carbon-aware-fl-results